In [ ]:
import os, tempfile, subprocess, shutil, time, json
from pathlib import Path
import torch
root = Path(tempfile.mkdtemp())
os.chdir(root)
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
%%writefile worker.py
import time
import os
import sys
from pathlib import Path
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(71)
torch.set_num_threads(1)
model = torch.nn.Linear(16, 4).to(device)
if sys.argv[1] == "short":
    with open("starts", "a") as log:
        print(time.monotonic(), file=log)
    with torch.inference_mode():
        model(torch.ones(1, 16, device=device)).cpu()
    print("Short inference job finished on", device, flush=True)
    sys.exit()

optimizer = torch.optim.SGD(model.parameters(), lr=.01, momentum=.9)
checkpoint = Path("checkpoint.pt")
start = 0
if sys.argv[1] == "resume":
    state = torch.load(checkpoint, map_location=device, weights_only=True)
    model.load_state_dict(state["model"])
    optimizer.load_state_dict(state["optimizer"])
    start = state["step"]
    print("Resuming after step", start, flush=True)

for step in range(start + 1, 101):
    generator = torch.Generator().manual_seed(step)
    batch = torch.randn(32, 16, generator=generator).to(device)
    target = batch[:, :4] * 2
    optimizer.zero_grad()
    (model(batch) - target).square().mean().backward()
    optimizer.step()
    if step == 1:
        Path("active").touch()
    time.sleep(0.05)  # Controlled pacing for repeatable mid-run submissions.
    if Path("pause").exists():
        torch.save(dict(step=step, model=model.state_dict(), optimizer=optimizer.state_dict()), "pending.pt")
        os.replace("pending.pt", checkpoint)
        sys.exit(75)
torch.save(model.state_dict(), sys.argv[1] + ".pt")

Path("finished").write_text(str(time.monotonic()))


In [ ]:
%%writefile scheduler.cpp
#include <sys/wait.h>
#include <unistd.h>
#include <cerrno>
#include <chrono>
#include <filesystem>
#include <fstream>
#include <iostream>
#include <stdexcept>
#include <thread>
namespace fs = std::filesystem;
using namespace std::chrono_literals;
pid_t launch(const char* mode) {
    pid_t pid = fork();
    if (pid < 0) throw std::runtime_error("fork failed");
    if (pid == 0) {
        execlp("python3", "python3", "worker.py", mode, nullptr);
        _exit(127);
    }
    return pid;
}
int status(pid_t pid, bool block) {
    int value;
    pid_t result;
    do { result = waitpid(pid, &value, block ? 0 : WNOHANG); } while (result < 0 && errno == EINTR);
    if (result < 0) throw std::runtime_error("waitpid failed");
    if (!result) return -1;
    return WIFEXITED(value) ? WEXITSTATUS(value) : 128;
}
void drain() {
    while (!fs::is_empty("queue")) {
        fs::path next;
        for (const auto& entry : fs::directory_iterator("queue"))
            if (next.empty() || entry.path() < next) next = entry.path();
        fs::remove(next);
        if (status(launch("short"), true) != 0) throw std::runtime_error("inference failed");
    }
}
int main(int argc, char** argv) {
    if (argc != 2) return 2;
    bool preempt = std::string(argv[1]) == "checkpoint";
    auto trainer = launch("train");
    int code;
    while ((code = status(trainer, false)) == -1) {
        if (preempt && !fs::is_empty("queue") && !fs::exists("pause")) std::ofstream{"pause"};
        std::this_thread::sleep_for(5ms);
    }
    if (code != 0 && code != 75) return 1;
    std::ofstream{"training_exit"} << code;
    drain();
    if (code == 75) {
        fs::remove("pause");
        if (status(launch("resume"), true) != 0) return 1;
    }
}


In [ ]:
subprocess.run(['g++','-O2','-std=c++17','scheduler.cpp','-o','scheduler'],check=True)

In [ ]:
results = {}
weights = {}
for policy in ['fcfs', 'checkpoint']:
    folder = Path(tempfile.mkdtemp(dir=root))
    (folder/'queue').mkdir()
    shutil.copy(root/'worker.py', folder)
    with subprocess.Popen([root/'scheduler',policy],cwd=folder) as proc:
        deadline = time.monotonic()+120
        while not (folder/'active').exists():
            if proc.poll() is not None or time.monotonic()>deadline:
                proc.kill()
                raise RuntimeError('Trainer failed to start')
            time.sleep(.01)
        started = time.monotonic()
        arrivals = []
        for index in range(2):
            time.sleep(.25)
            arrivals.append(time.monotonic())
            (folder/'queue'/str(index)).touch()
        if proc.wait(timeout=180):
            raise RuntimeError('Worker failed')
    starts = [float(x) for x in (folder/'starts').read_text().splitlines()]
    assert len(starts)==len(arrivals)
    exit_code = int((folder/'training_exit').read_text())
    assert exit_code == (75 if policy=='checkpoint' else 0)
    results[policy] = dict(short_wait_seconds=[round(s-a,3) for s,a in zip(starts,arrivals)],
                           training_elapsed_seconds=round(float((folder/'finished').read_text())-started,3))
    weights[policy] = torch.load(folder/('resume.pt' if exit_code else 'train.pt'), map_location='cpu', weights_only=True)
for key,value in weights['fcfs'].items():
    torch.testing.assert_close(value,weights['checkpoint'][key])
print(json.dumps(results,indent=2))
print('PASS: both queued jobs ran and final weights match.')
(root/'results.json').write_text(json.dumps(results,indent=2))

In [ ]:
from google.colab import files
files.download(str(root/'results.json'))